# Pilot Model v2 — reproducible pipeline

Полный путь: model-ready data → конфигурация признаков → temporal validation → final fit Jan–Oct → Nov–Dec inference → правило route 5 → submission. Единица наблюдения — `(route, date, hour)`, target — `boardings`. Здесь используются только готовые CSV; raw labels, grid и features заново не строятся. Конфигурация v2 является **кандидатом** и не меняет accepted feature registry.

## Environment & Data Setup

Загружаем два model-ready CSV согласно `data/processed/README_model_ready.md`: UTF-8, разделитель `;`, `date` как datetime. Путь определяется от текущего каталога notebook или корня репозитория; в Colab поддержан путь v1 через Google Drive. Версии библиотек выводятся для воспроизводимости.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm
from IPython.display import display
from lightgbm import LGBMRegressor

def find_data_root():
    candidates = [parent / 'data' for parent in (Path.cwd(), *Path.cwd().parents)]
    candidates.append(Path('/content/drive/MyDrive/MT-Hackathon-2026/data'))
    for candidate in candidates:
        if (candidate / 'processed' / 'model_ready_train_2025_jan_oct.csv').is_file():
            return candidate
    raise FileNotFoundError('Model-ready data directory not found')

if Path('/content').exists() and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

DATA_ROOT = find_data_root()
PROCESSED_DIR = DATA_ROOT / 'processed'
TRAIN_PATH = PROCESSED_DIR / 'model_ready_train_2025_jan_oct.csv'
FORECAST_PATH = PROCESSED_DIR / 'model_ready_forecast_2025_nov_dec.csv'
SUBMISSION_PATH = PROCESSED_DIR / 'pilot_model_v2_submission.csv'

train = pd.read_csv(TRAIN_PATH, sep=';', encoding='utf-8', parse_dates=['date'])
forecast = pd.read_csv(FORECAST_PATH, sep=';', encoding='utf-8', parse_dates=['date'])
print('pandas:', pd.__version__, 'LightGBM:', lightgbm.__version__)
print('train rows:', len(train), 'forecast rows:', len(forecast))

Mounted at /content/drive
pandas: 2.2.3 LightGBM: 4.6.0
train rows: 72960 forecast rows: 14640


## Data Contract / Sanity Checks

Проверки останавливают pipeline при изменении входного контракта: нужны ровно указанные столбцы, все дни периода, 10 маршрутов × 24 часа, уникальные ключи и отсутствие пропусков. Forecast не должен содержать target. Это защищает сравнение и submission от тихого сдвига схемы или сетки.

In [2]:
ROUTES = (1, 5, 7, 11, 12, 17, 25, 26, 28, 50)
KEYS = ['route', 'date', 'hour']
CORE_FEATURES = (
    'route', 'weekday', 'hour', 'route_hour', 'hour_weekend', 'is_night'
)
CALENDAR_CANDIDATES = (
    'is_day_off', 'is_official_holiday', 'is_transferred_day_off',
    'is_transferred_workday', 'is_shortened_workday'
)
V2_FEATURES = CORE_FEATURES + CALENDAR_CANDIDATES
FORECAST_COLUMNS = [
    'route', 'date', 'hour', 'weekday', 'route_hour', 'hour_weekend',
    'is_night', *CALENDAR_CANDIDATES
]
TRAIN_COLUMNS = FORECAST_COLUMNS + ['boardings']

if train.columns.tolist() != TRAIN_COLUMNS:
    raise ValueError('Train schema differs from model-ready contract')
if forecast.columns.tolist() != FORECAST_COLUMNS:
    raise ValueError('Forecast schema differs from model-ready contract')
if train.drop(columns='boardings').columns.tolist() != forecast.columns.tolist():
    raise ValueError('Train and forecast feature columns differ')
if set(train.columns) - set(KEYS) - {'boardings'} != set(V2_FEATURES) - set(KEYS):
    raise ValueError('Unexpected feature columns')

def check_grid(frame, start, end, expected_rows):
    dates = pd.date_range(start, end, freq='D')
    if len(frame) != expected_rows or frame[KEYS].isna().any().any():
        raise ValueError('Incorrect row count or missing keys')
    if frame.isna().any().any():
        raise ValueError('Missing values in model-ready data')
    if not pd.api.types.is_datetime64_any_dtype(frame['date']):
        raise TypeError('date must be datetime')
    if frame[KEYS].duplicated().any():
        raise ValueError('Duplicate route/date/hour keys')
    if set(frame['route']) != set(ROUTES) or set(frame['hour']) != set(range(24)):
        raise ValueError('Unexpected route or hour values')
    if set(frame['date']) != set(dates):
        raise ValueError('Incomplete or unexpected date period')
    if len(frame) != len(ROUTES) * len(dates) * 24:
        raise ValueError('Incomplete route/date/hour grid')
    if not all(pd.api.types.is_numeric_dtype(frame[col]) for col in frame.columns if col != 'date'):
        raise TypeError('Non-numeric feature or target')
    if not frame['weekday'].eq(frame['date'].dt.dayofweek).all():
        raise ValueError('weekday does not match date')
    if not frame['route_hour'].eq(frame['route'] * 24 + frame['hour']).all():
        raise ValueError('route_hour encoding differs from v1')
    if not frame['hour_weekend'].eq(frame['hour'] * 2 + frame['weekday'].ge(5).astype('int8')).all():
        raise ValueError('hour_weekend encoding differs from v1')
    if not frame['is_night'].eq(frame['hour'].between(0, 5).astype('int8')).all():
        raise ValueError('is_night encoding differs from v1')
    for col in ('is_night', *CALENDAR_CANDIDATES):
        if not frame[col].isin((0, 1)).all():
            raise ValueError(f'{col} must be binary')

check_grid(train, '2025-01-01', '2025-10-31', 72_960)
check_grid(forecast, '2025-11-01', '2025-12-31', 14_640)
if 'boardings' in forecast.columns:
    raise ValueError('Forecast must not contain target')
if train['boardings'].lt(0).any():
    raise ValueError('Negative training target')
print('Model-ready data contract passed.')

Model-ready data contract passed.


## Feature Configuration

`CORE_FEATURES` — frozen reference v1. `CALENDAR_CANDIDATES` добавлены только в `V2_FEATURES` для контролируемого сравнения. Пять полей v1 остаются категориальными с фиксированными категориями, совместимыми с v1; `is_night` и calendar flags остаются числовыми. Никакие weather или historical H19–H27 признаки не используются. Одинаковая функция preprocessing применяется к обеим конфигурациям.

In [3]:
CATEGORICAL_FEATURES = ('route', 'weekday', 'hour', 'route_hour', 'hour_weekend')
CATEGORIES = {
    'route': ROUTES,
    'weekday': tuple(range(7)),
    'hour': tuple(range(24)),
    'route_hour': tuple(route * 24 + hour for route in ROUTES for hour in range(24)),
    'hour_weekend': tuple(range(48)),
}
MODEL_PARAMS = {
    'objective': 'regression_l1',
    'n_estimators': 300,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.1,
    'random_state': 42,
    'n_jobs': 4,
    'verbosity': -1,
}

def make_features(rows, feature_set):
    features = rows.loc[:, list(feature_set)].copy()
    for col in CATEGORICAL_FEATURES:
        features[col] = pd.Categorical(features[col], categories=CATEGORIES[col])
        if features[col].isna().any():
            raise ValueError(f'Unknown category in {col}')
    for col in set(feature_set) - set(CATEGORICAL_FEATURES):
        features[col] = features[col].astype('int8')
    return features

for feature_set in (CORE_FEATURES, V2_FEATURES):
    for frame in (train, forecast):
        prepared = make_features(frame, feature_set)
        if prepared.columns.tolist() != list(feature_set):
            raise ValueError('Feature order mismatch')
        if any(not isinstance(prepared[col].dtype, pd.CategoricalDtype) for col in CATEGORICAL_FEATURES):
            raise TypeError('Categorical preprocessing mismatch')
print('Feature sets and shared preprocessing passed.')

Feature sets and shared preprocessing passed.


## Validation Strategy

Основная проверка имитирует прогноз из конца августа: обучение только Jan–Aug, оценка Sep–Oct. У обоих вариантов один split, один frozen LightGBM, preprocessing и postprocessing. Target Sep–Oct не попадает в признаки или обучение; он используется после получения predictions только для WAPE. WAPE = сумма абсолютных ошибок / сумма фактических посадок, WAPE-score = `max(0, 1 − WAPE)`. Для route 5 с нулевой суммой target per-route WAPE не определён (`NaN`).

In [4]:
validation_train = train.loc[train['date'] <= '2025-08-31'].copy()
validation_holdout = train.loc[train['date'] >= '2025-09-01'].copy()
if len(validation_train) != 58_320 or len(validation_holdout) != 14_640:
    raise ValueError('Temporal split has unexpected size')
if validation_train['date'].max() >= validation_holdout['date'].min():
    raise ValueError('Temporal split overlaps')

def make_model():
    return LGBMRegressor(**MODEL_PARAMS)

def postprocess_predictions(raw_predictions, routes):
    prediction = np.floor(np.maximum(np.asarray(raw_predictions), 0) + 0.5).astype('int64')
    prediction[np.asarray(routes) == 5] = 0
    return prediction

def calculate_metrics(actual, prediction, routes):
    actual = np.asarray(actual)
    prediction = np.asarray(prediction)
    total_actual = actual.sum()
    if total_actual <= 0:
        raise ValueError('Overall WAPE is undefined for zero total actual')
    absolute_error = np.abs(actual - prediction)
    wape = absolute_error.sum() / total_actual
    route_frame = pd.DataFrame({
        'route': np.asarray(routes), 'actual': actual, 'absolute_error': absolute_error
    })
    route_metrics = route_frame.groupby('route').agg(
        actual=('actual', 'sum'), absolute_error=('absolute_error', 'sum')
    ).reindex(ROUTES)
    route_metrics['WAPE'] = route_metrics['absolute_error'] / route_metrics['actual'].replace(0, np.nan)
    return wape, max(0.0, 1.0 - wape), route_metrics.reset_index()

## Jan–Aug → Sep–Oct controlled validation

Обучаем reference и v2 candidate независимо с теми же параметрами. После одинакового округления прогноз route 5 устанавливается в ноль. `Δ WAPE = WAPE(v2) − WAPE(core)`: отрицательное значение означает улучшение на этом holdout, но само по себе не делает calendar group accepted feature set.

In [5]:
validation_results = []
route_results = {}
for name, feature_set in [('core_reference', CORE_FEATURES), ('v2_candidate', V2_FEATURES)]:
    model = make_model()
    model.fit(
        make_features(validation_train, feature_set), validation_train['boardings'],
        categorical_feature=list(CATEGORICAL_FEATURES),
    )
    prediction = postprocess_predictions(
        model.predict(make_features(validation_holdout, feature_set)),
        validation_holdout['route'],
    )
    wape, score, per_route = calculate_metrics(
        validation_holdout['boardings'], prediction, validation_holdout['route']
    )
    validation_results.append({'configuration': name, 'WAPE': wape, 'WAPE-score': score})
    route_results[name] = per_route.set_index('route')['WAPE']

validation_summary = pd.DataFrame(validation_results)
delta_wape = (validation_summary.loc[1, 'WAPE'] - validation_summary.loc[0, 'WAPE'])
route_wape = pd.DataFrame(route_results)
route_wape['delta_WAPE_v2_minus_core'] = route_wape['v2_candidate'] - route_wape['core_reference']
display(validation_summary.round(6))
print(f'Delta WAPE (v2 - core): {delta_wape:+.6f}')
display(route_wape.round(6))
print('Calendar group status: candidate representation; accepted feature registry unchanged.')

,configuration,WAPE,WAPE-score
0,core_reference,0.123580,0.876420
1,v2_candidate,0.120922,0.879078


Delta WAPE (v2 - core): -0.002658


,core_reference,v2_candidate,delta_WAPE_v2_minus_core
route,,,
1,0.109409,0.104101,-0.005309
5,NaN,NaN,NaN
7,0.157333,0.152735,-0.004598
11,0.105455,0.104823,-0.000632
12,0.088066,0.085928,-0.002138
17,0.085894,0.084201,-0.001693
25,0.199599,0.189324,-0.010275
26,0.125388,0.120151,-0.005237
28,0.153739,0.149810,-0.003929


Calendar group status: candidate representation; accepted feature registry unchanged.


## Final Training Jan–Oct

**Pilot v2 candidate configuration:** обучаем frozen LightGBM с `V2_FEATURES` на всех model-ready строках Jan–Oct. Sep–Oct target теперь используется только в final fit после завершённой validation; по нему не подбирались параметры модели или признаки. Это не изменение accepted feature registry.

In [6]:
final_model = make_model()
final_model.fit(
    make_features(train, V2_FEATURES), train['boardings'],
    categorical_feature=list(CATEGORICAL_FEATURES),
)
print('Final v2 candidate model trained on', len(train), 'Jan-Oct rows.')

Final v2 candidate model trained on 72960 Jan-Oct rows.


## Nov–Dec Inference

Берём **непосредственно** `model_ready_forecast_2025_nov_dec.csv` в исходном порядке. Контракт периода, полной сетки и схемы уже проверен; grid не пересоздаётся. Применяем те же категории и округление `floor(max(raw, 0) + 0.5)`, затем принудительно устанавливаем `prediction = 0` для route 5 согласно inference policy.

In [7]:
if forecast.columns.tolist() != FORECAST_COLUMNS or 'boardings' in forecast:
    raise ValueError('Forecast feature schema or target presence changed')
inference = forecast.loc[:, KEYS].copy()
inference['prediction'] = postprocess_predictions(
    final_model.predict(make_features(forecast, V2_FEATURES)), forecast['route']
)
if not inference.loc[inference['route'].eq(5), 'prediction'].eq(0).all():
    raise ValueError('Route 5 policy failed')
print('Predictions generated:', len(inference))

Predictions generated: 14640


## Submission

Сохраняем ровно `route;date;hour;prediction` как UTF-8 CSV с `;`. `inference` наследует исходный порядок forecast, без сортировки и изменения ключей. Это локальный pilot v2 candidate artifact.

In [8]:
submission = inference.loc[:, ['route', 'date', 'hour', 'prediction']].copy()
submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
submission.to_csv(SUBMISSION_PATH, sep=';', index=False, encoding='utf-8')
print('Submission saved to:', SUBMISSION_PATH)

Submission saved to: /content/drive/MyDrive/MT-Hackathon-2026/data/processed/pilot_model_v2_submission.csv


## Final Sanity Checks

Повторно читаем сохранённый файл, чтобы проверить именно submission artifact: число строк, полный Nov–Dec период, маршруты и часы, уникальность ключей, точный порядок forecast, целочисленные неотрицательные прогнозы без NaN и нули route 5. Схема forecast исключает target и observed weather; список `V2_FEATURES` также явно исключает weather.

In [9]:
saved = pd.read_csv(SUBMISSION_PATH, sep=';', encoding='utf-8')
if saved.columns.tolist() != ['route', 'date', 'hour', 'prediction']:
    raise ValueError('Saved submission columns or order differ')
if len(saved) != 14_640 or saved[KEYS].duplicated().any():
    raise ValueError('Saved submission row count or keys are invalid')
if saved.isna().any().any():
    raise ValueError('Saved submission contains NaN')
if set(saved['route']) != set(ROUTES) or set(saved['hour']) != set(range(24)):
    raise ValueError('Saved submission routes or hours differ')
if set(saved['date']) != set(pd.date_range('2025-11-01', '2025-12-31').strftime('%Y-%m-%d')):
    raise ValueError('Saved submission period differs')
if saved['prediction'].lt(0).any() or not pd.api.types.is_integer_dtype(saved['prediction']):
    raise ValueError('Saved predictions must be non-negative integers')
if not saved.loc[saved['route'].eq(5), 'prediction'].eq(0).all():
    raise ValueError('Saved route 5 predictions must be zero')
expected_keys = forecast[KEYS].copy()
expected_keys['date'] = expected_keys['date'].dt.strftime('%Y-%m-%d')
if not saved[KEYS].equals(expected_keys.reset_index(drop=True)):
    raise ValueError('Saved keys or row order differ from forecast input')
if 'boardings' in forecast or any('weather' in col.lower() for col in V2_FEATURES):
    raise ValueError('Target or weather must not be used for inference')
print('Final sanity checks passed: 14,640 Nov-Dec rows, exact keys/order, valid integer predictions, route 5 zero, no target/weather input.')

Final sanity checks passed: 14,640 Nov-Dec rows, exact keys/order, valid integer predictions, route 5 zero, no target/weather input.
